# 2 卷积和池化
## 2.1 理论计算题

### 1. 计算卷积输出特征图尺寸
**已知条件：**
输入图像：$3 \times 32 \times 32$（通道×高×宽）
卷积核数量：16 个，卷积核空间尺寸：$5 \times 5$
填充 $P=2$，步幅 $S=2$

卷积输出尺寸公式：
$$
H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1
$$
$$
W_{out} = \left\lfloor \frac{W_{in} + 2P - K}{S} \right\rfloor + 1
$$

代入计算：
$$
H_{out} = \left\lfloor \frac{32 + 2\times2 - 5}{2} \right\rfloor + 1 = 16
$$
$$
W_{out} = \left\lfloor \frac{32 + 2\times2 - 5}{2} \right\rfloor + 1 = 16
$$

输出通道数等于卷积核个数：$16$

**结论：**
输出特征图尺寸为 $\boldsymbol{16 \times 16 \times 16}$（通道 × 高 × 宽）。

### 2. 计算单像素乘法操作次数
单个输出像素由输入多通道特征与卷积核逐元素相乘后求和得到。

乘法总次数：
$$
3 \times 5 \times 5 = 75
$$

**结论：**
单个输出通道的一个像素，需要进行 $\boldsymbol{75}$ 次乘法操作。



In [1]:
import numpy as np

def max_pool2d_manual(x, kernel_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    x: 输入张量，shape (batch, channels, height, width)
    kernel_size: 池化窗口大小 (kh, kw) 或 int
    stride: 步幅 (sh, sw) 或 int，默认等于 kernel_size
    padding: 填充数 (ph, pw) 或 int
    """
    if stride is None:
        stride = kernel_size
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    batch, ch, h, w = x.shape
    # 填充
    x_pad = np.pad(x, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant')
    h_out = (h + 2*ph - kh) // sh + 1
    w_out = (w + 2*pw - kw) // sw + 1
    out = np.zeros((batch, ch, h_out, w_out))

    for b in range(batch):
        for c in range(ch):
            for i in range(h_out):
                for j in range(w_out):
                    h_start = i * sh
                    h_end = h_start + kh
                    w_start = j * sw
                    w_end = w_start + kw
                    window = x_pad[b, c, h_start:h_end, w_start:w_end]
                    out[b, c, i, j] = np.max(window)
    return out

# 示例
if __name__ == "__main__":
    x = np.random.randn(2, 3, 32, 32)
    out = max_pool2d_manual(x, kernel_size=2, stride=2, padding=0)
    print("输出形状:", out.shape)  # (2,3,16,16)

输出形状: (2, 3, 16, 16)


# 3 LeNet, AlexNet, VGG 和 NiN
## 3.1 理论计算题

### 1. 单个 5×5 卷积层参数量
**已知条件：**
输入、输出通道数均为 $C$，卷积层无偏置项。

卷积参数量公式：
$$
参数量 = 输入通道数 \times 卷积核高 \times 卷积核宽 \times 输出通道数
$$

代入计算：
$$
C \times 5 \times 5 \times C = 25C^2
$$

**结论：**
单个 $5 \times 5$ 卷积层参数量为 $\boldsymbol{25C^2}$。


### 2. 两层串联 3×3 卷积层总参数量
单层 $3 \times 3$ 卷积参数量：
$$
C \times 3 \times 3 \times C = 9C^2
$$

两层串联总参数量：
$$
9C^2 + 9C^2 = 18C^2
$$

**结论：**
两层串联 $3 \times 3$ 卷积层总参数量为 $\boldsymbol{18C^2}$。


In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        """
        定义NiN块
        结构：普通卷积 + ReLU + 1×1卷积 + ReLU + 1×1卷积 + ReLU
        :param in_channels: 输入通道数
        :param out_channels: 输出通道数
        :param kernel_size: 主卷积核尺寸
        :param stride: 主卷积步幅
        :param padding: 主卷积填充
        """
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,
                      stride=stride, padding=padding),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

# 测试运行
if __name__ == "__main__":
    nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    x = torch.randn(2, 3, 32, 32)
    out = nin_block(x)
    print(f"输入尺寸: {x.shape}")
    print(f"输出尺寸: {out.shape}")

输入尺寸: torch.Size([2, 3, 32, 32])
输出尺寸: torch.Size([2, 16, 32, 32])


# 4 Inception, 批量归一化和残差网络
## 4.1 理论计算题
**已知条件：**
样本值：$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8$
缩放参数 $\gamma=2$，平移参数 $\beta=1$，常数 $\epsilon=0$

批量归一化公式：
$$
\mu = \frac{1}{m}\sum_{i=1}^m x_i
$$
$$
\sigma^2 = \frac{1}{m}\sum_{i=1}^m (x_i-\mu)^2
$$
$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
$$
y_i = \gamma \cdot \hat{x}_i + \beta
$$

代入计算：
$$
\mu = \frac{2+4+6+8}{4} = 5
$$
$$
\sigma^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4} = 5
$$

归一化并缩放平移：
$$
y_1 = 2 \cdot \frac{2-5}{\sqrt{5}} + 1 \approx -1.683
$$
$$
y_2 = 2 \cdot \frac{4-5}{\sqrt{5}} + 1 \approx 0.106
$$
$$
y_3 = 2 \cdot \frac{6-5}{\sqrt{5}} + 1 \approx 1.894
$$
$$
y_4 = 2 \cdot \frac{8-5}{\sqrt{5}} + 1 \approx 4.683
$$

**结论：**
$y_1\approx\boldsymbol{-1.683},\ y_2\approx\boldsymbol{0.106},\ y_3\approx\boldsymbol{1.894},\ y_4\approx\boldsymbol{4.683}$



In [4]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        """
        自定义残差块
        :param in_channels: 输入通道数
        :param out_channels: 输出通道数
        :param use_1x1conv: 是否使用1×1卷积做维度匹配
        :param stride: 卷积步幅
        """
        super(Residual, self).__init__()
        # 主分支：两层3×3卷积 + 批量归一化（这里补上了 out_channels）
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 旁路1×1卷积（维度对齐）
        self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride) if use_1x1conv else None

    def forward(self, x):
        # 主分支前向传播
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # 旁路维度对齐
        if self.conv3 is not None:
            x = self.conv3(x)

        # 残差连接相加
        out += x
        out = self.relu(out)
        return out

# 测试运行
if __name__ == "__main__":
    # 维度一致场景
    res_block1 = Residual(64, 64, use_1x1conv=False)
    x1 = torch.randn(1, 64, 32, 32)
    print(f"维度一致 | 输入: {x1.shape} 输出: {res_block1(x1).shape}")

    # 维度变化场景
    res_block2 = Residual(64, 128, use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 64, 32, 32)
    print(f"维度变化 | 输入: {x2.shape} 输出: {res_block2(x2).shape}")

维度一致 | 输入: torch.Size([1, 64, 32, 32]) 输出: torch.Size([1, 64, 32, 32])
维度变化 | 输入: torch.Size([1, 64, 32, 32]) 输出: torch.Size([1, 128, 16, 16])


# 5 图像增广，微调和样式迁移
## 5.1 理论计算题
1. 微调学习率设置原因

预训练网络底层已经学习到边缘、纹理、轮廓等通用视觉特征，这类特征具备普适性，因此对底层特征提取层设置较小学习率（或冻结参数），防止破坏已有优质特征。
网络顶层输出层针对原数据集训练，和新任务无关，需要重新学习，因此对顶层输出层设置较大学习率，加快参数收敛。

2. 小样本 + 相似数据集的微调策略

当目标数据集样本少、且与源数据集高度相似时，极易发生过拟合。
优先策略：冻结全部特征提取层，仅训练最后分类层；
全程使用小学习率，不改动预训练主干权重；
同时搭配图像增广、正则化等方式抑制过拟合。

In [8]:
from torchvision import transforms
from PIL import Image
import numpy as np

# 构建图像增广流水线（严格按照题目要求）
aug_pipeline = transforms.Compose([
    # 随机裁剪，面积比例 0.08 ~ 1.0，缩放到 224×224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 随机调整亮度、对比度、饱和度，变化范围 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 转为PyTorch张量
    transforms.ToTensor()
])

# 测试运行（使用numpy数组模拟图像，无需本地图片）
if __name__ == "__main__":
    # 生成 256*256 三通道模拟图像（numpy数组，符合PIL输入格式）
    fake_img_np = np.random.randint(0, 255, size=(256, 256, 3), dtype=np.uint8)
    # 转为 PIL Image
    fake_img = Image.fromarray(fake_img_np)
    
    # 执行图像增广
    aug_img = aug_pipeline(fake_img)
    print(f"增广后图像张量尺寸: {aug_img.shape}")

增广后图像张量尺寸: torch.Size([3, 224, 224])


# 6 目标检测, 计算机视觉训练技巧
## 6.1 理论计算题
**已知条件：**
真实框 $A=[10,10,50,50]$
预测框 $B=[30,30,70,70]$

IoU 计算公式：
$$
IoU = \frac{交集面积}{并集面积}
$$

分步计算：
$$
x_{min}=\max(10,30)=30,\quad y_{min}=\max(10,30)=30
$$
$$
x_{max}=\min(50,70)=50,\quad y_{max}=\min(50,70)=50
$$

交集面积：
$$
S_{inter} = (50-30) \times (50-30) = 400
$$

框A面积：
$$
S_A = (50-10) \times (50-10) = 1600
$$

框B面积：
$$
S_B = (70-30) \times (70-30) = 1600
$$

并集面积：
$$
S_{union} = 1600 + 1600 - 400 = 2800
$$

IoU：
$$
IoU = \frac{400}{2800} = \frac{1}{7} \approx 0.1429
$$

**结论：**
两框交并比 $\boldsymbol{IoU=\dfrac{1}{7} \approx 0.1429}$


In [6]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, num_classes, eps=0.1):
    """
    实现标签平滑交叉熵损失
    :param logits: 模型原始输出 [N, num_classes]
    :param labels: 真实标签 [N]
    :param num_classes: 分类总数 K
    :param eps: 标签平滑系数
    :return: 批次平均损失
    """
    batch_size = logits.shape[0]
    # 初始化软标签
    soft_target = torch.full((batch_size, num_classes), eps / (num_classes - 1), device=logits.device)
    # 真实类别赋值 1-eps
    soft_target.scatter_(dim=1, index=labels.unsqueeze(1), value=1.0 - eps)

    # 计算交叉熵损失
    log_prob = F.log_softmax(logits, dim=1)
    loss = -torch.sum(soft_target * log_prob, dim=1)
    return torch.mean(loss)

# 测试运行
if __name__ == "__main__":
    # 模拟5分类任务
    logits = torch.randn(8, 5)
    labels = torch.tensor([0, 1, 2, 3, 4, 0, 1, 2])
    loss = label_smoothing_cross_entropy(logits, labels, num_classes=5, eps=0.1)
    print(f"标签平滑交叉熵损失值: {loss:.4f}")

标签平滑交叉熵损失值: 1.4486
